# 3 — Phase–amplitude coupling in IT local field potentials

Local field potentials from 17 sessions, recorded alongside the spiking data
in notebook 2. The question is whether the phase of a low-frequency rhythm
(4–20 Hz) organises the amplitude of gamma (30–130 Hz), and whether that
coupling depends on which category of image was shown.

Two estimators are computed for each session and category, on the same
filtered data:

- **Tort MI** (`idpac=(2,0,0)`) — Kullback–Leibler divergence of the
  amplitude-by-phase histogram from uniform.
- **Canolty MVL** (`idpac=(1,0,0)`) — mean vector length of the complex
  amplitude series.

Welch PSD per condition is plotted alongside, as a sanity check that the
gamma band being measured has power in it.

**Data.** Needs `data_LFP.mat`, which is not in the repository — see the
README.

In [ ]:
import os
from pathlib import Path

# The recordings are hundreds of megabytes and are not committed. Point
# SPIKE_DATA_DIR at wherever they live, or drop them in ./data/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = Path(os.environ.get('SPIKE_DATA_DIR') or ROOT / 'data')

if not DATA.is_dir():
    raise SystemExit(
        'No data directory at %s.\n'
        'Set SPIKE_DATA_DIR or create ./data/ — see the README.' % DATA)
print('data:', DATA)


## 1 — Per-session comodulograms

**Defect: PAC was computed on the trial average.** The original reduced each
condition to one mean waveform and ran the estimator on that:

```python
avg_sig = sig[:, mask].mean(axis=1)
sig2d   = avg_sig[np.newaxis, :]
mi      = pac_mi.filterfit(FS, sig2d)
```

Averaging across trials keeps only the part of the signal that is
phase-locked to stimulus onset and cancels everything that is not. Gamma
amplitude is largely *induced* — it is time-locked but not phase-locked — so
trial averaging removes most of the signal the analysis is trying to
measure, and what survives is dominated by the evoked potential. The
comodulogram of the average is not the average comodulogram.

tensorpac accepts an `(n_epochs, n_times)` array and returns one map per
epoch, so the fix is to hand it the trials and average the maps. The same
correction applies to the PSD, which is now the mean of the per-trial
spectra rather than the spectrum of the mean.

**Defect: no surrogate correction.** Raw MI and MVL are both biased upward
by signal length, filter bandwidth and non-sinusoidal waveform shape, so an
uncorrected comodulogram has no zero point — it cannot distinguish coupling
from the bias floor. `SURROGATES = True` switches both estimators to
z-scores against time-shifted surrogates (`idpac=(·,3,0)`). It is off by
default because 200 permutations per session and category is roughly an
hour on this data.

The per-session figures are limited to session 0; set `PLOT_SESSION = None`
for all 17. The cross-session average is the next section.

In [ ]:
"""Phase-amplitude coupling in IT LFPs, by stimulus category."""
import numpy as np
import mat73
import matplotlib.pyplot as plt
from tensorpac import Pac
from mne.time_frequency import psd_array_welch
from scipy.signal import detrend

MAT_FILE = DATA / 'data_LFP.mat'
FS = 900 / 0.9                          # 900 samples over 0.9 s
PHASE_BAND = np.linspace(4, 20, 17)     # phase-giving frequencies (Hz)
AMP_BAND = np.linspace(30, 130, 21)     # amplitude-receiving frequencies
CATEGORIES = ['Face', 'Body', 'Natural', 'Artificial']
MIN_TRIALS = 5
N_AVG_SESS = 17

# Surrogate correction: z-score against time-shifted surrogates instead of
# raw MI/MVL. Roughly an hour for all sessions, hence off by default.
SURROGATES = False
N_PERM = 200
SURR = 3 if SURROGATES else 0

# Per-session figures for this session only; None plots all 17.
PLOT_SESSION = 0
FACE_LABELS = {1,2,3,4,5,6,7,8,9,10,11,12,13,15,17,18,27,75,76,77,78,79,80,83,
               102,103,104,105,106,107,110,129,130,131,132,133,134,137,
               156,157,158,159,160,161,162,163,164,165,166,167,168,169,170}
BODY_LABELS = {14,16,19,20,21,22,23,24,25,26,28,29,30,31,33,34,35,36,37,
               84,85,86,87,88,89,111,112,113,114,115,116,138,139,140,
               141,142,143}
NATURAL_LABELS = {32,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,72,73,
                  90,91,92,93,94,95,117,118,119,120,121,122,144,145,146,147,
                  148,149}
ARTIFICIAL_LABELS = {55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,74,
                   96,97,98,99,100,101,123,124,125,126,127,128,150,151,
                   152,153,154,155}
LABELS_MAP = {**{lbl: 'Face' for lbl in FACE_LABELS},
              **{lbl: 'Body' for lbl in BODY_LABELS},
              **{lbl: 'Natural' for lbl in NATURAL_LABELS},
              **{lbl: 'Artificial' for lbl in ARTIFICIAL_LABELS}}
mat = mat73.loadmat(MAT_FILE)
data_it_list = mat['data_LFP_it']['data_it']
cm_list = mat['data_LFP_it']['cm']
mi_store = {cat: [] for cat in CATEGORIES}
ca_store = {cat: [] for cat in CATEGORIES}
for sess_idx, (data_it, cm_vec) in enumerate(zip(data_it_list, cm_list)):
    if sess_idx >= N_AVG_SESS:
        break
    sig = np.asarray(data_it)
    if sig.ndim != 2:
        raise ValueError(f"Session {sess_idx}: data_it must be 2D")
    if sig.shape[0] < sig.shape[1]:
        sig = sig.T
    n_samples, n_trials = sig.shape
    print(f"\nSession {sess_idx}: n_samples = {n_samples}, n_trials = {n_trials}, FS = {FS:.2f} Hz")
    cm_arr = np.asarray(cm_vec).squeeze()
    if cm_arr.ndim != 1:
        cm_arr = cm_arr.ravel()
    labels = np.array([LABELS_MAP.get(int(x), 'Unknown') for x in cm_arr], dtype=object)
    if labels.size != n_trials:
        n_min = min(labels.size, n_trials)
        sig = sig[:, :n_min]
        labels = labels[:n_min]
        n_samples, n_trials = sig.shape
    for cat in CATEGORIES:
        mask = (labels == cat)
        n_cat = mask.sum()
        if n_cat < MIN_TRIALS:
            print(f"  {cat}: only {n_cat} trials (skipped)")
            continue
        # DEFECT. The original averaged the trials and ran PAC on the one
        # mean waveform. That keeps only the phase-locked (evoked) part of
        # the signal, and gamma amplitude is largely induced — not
        # phase-locked to onset — so trial averaging removes most of what
        # is being measured. tensorpac takes (n_epochs, n_times) and
        # returns one map per epoch; averaging the maps is the estimator
        # the section wants.
        trials = detrend(sig[:, mask], axis=0).T      # (n_cat, n_samples)

        pac_mi = Pac(idpac=(2, SURR, 0), f_pha=PHASE_BAND, f_amp=AMP_BAND,
                     dcomplex='wavelet')
        pac_ca = Pac(idpac=(1, SURR, 0), f_pha=PHASE_BAND, f_amp=AMP_BAND,
                     dcomplex='wavelet')
        mi = pac_mi.filterfit(FS, trials, n_perm=N_PERM).mean(-1)
        ca = pac_ca.filterfit(FS, trials, n_perm=N_PERM).mean(-1)
        mi_store[cat].append(mi)
        ca_store[cat].append(ca)

        # Mean of the per-trial spectra, for the same reason.
        psd, freqs = psd_array_welch(trials, sfreq=FS, fmin=0.5, fmax=150,
                                     n_fft=min(2048, n_samples), verbose=False)
        psd = psd.mean(axis=0)
        print(f"  {cat}: {n_cat} trials, PAC + PSD done")

        if PLOT_SESSION is not None and sess_idx != PLOT_SESSION:
            continue

        plt.figure(figsize=(5, 4))
        pac_mi.comodulogram(mi, title=f'S{sess_idx} {cat} - Tort MI')
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(5, 4))
        pac_ca.comodulogram(ca, title=f'S{sess_idx} {cat} - Canolty MVL')
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(5, 3))
        plt.semilogy(freqs, psd)
        plt.xlabel('Frequency (Hz)')
        plt.ylabel('PSD (power/Hz)')
        plt.title(f'S{sess_idx} {cat} - Welch PSD')
        plt.tight_layout()
        plt.show()

n_done = {cat: len(v) for cat, v in mi_store.items()}
print('sessions contributing per category:', n_done)


## 2 — Grand average across sessions

**Defect: the accumulated results were never used.** `mi_store` and
`ca_store` were filled for all 17 sessions and then never read — the
notebook ended after the last per-session figure. This is the plot they were
being collected for, and it is the one worth reading: a coupling peak that
survives averaging over 17 sessions is a different claim from a peak in one
of them.

In [ ]:
# DEFECT. mi_store and ca_store were filled for all 17 sessions and then
# never read. This is the figure they were accumulated for.
pac_ref = Pac(idpac=(2, 0, 0), f_pha=PHASE_BAND, f_amp=AMP_BAND,
              dcomplex='wavelet')
unit = 'z-score vs surrogates' if SURROGATES else 'raw (uncorrected)'

for store, label in ((mi_store, 'Tort MI'), (ca_store, 'Canolty MVL')):
    have = [c for c in CATEGORIES if store[c]]
    if not have:
        print('nothing to average for', label)
        continue
    plt.figure(figsize=(4 * len(have), 3.8))
    for i, cat in enumerate(have, start=1):
        plt.subplot(1, len(have), i)
        pac_ref.comodulogram(
            np.mean(store[cat], axis=0),
            title=f'{cat} (n = {len(store[cat])})',
            colorbar=(i == len(have)))
    plt.suptitle(f'Grand-average comodulogram - {label}, {unit}')
    plt.tight_layout()
    plt.show()
